# Evaluasi OOD: classifier Indonesia pada foto CGMacros

Notebook lokal untuk Sprint 2: **[CV] Run Indonesian classifier on CGMacros photos — measure OOD gap**.

Alur: foto sebelum makan → MobileNetV2 10 kelas → lookup makro TKPI → MAE terhadap label CGMacros → perbandingan B0 pada meal yang sama.

**Mulai:** pilih kernel **GlucoSight CV (NumPy 1.26)** / Python 3.10–3.12 dalam virtual environment, jalankan sel dari atas. Default MAX_MEALS = 32 untuk mencoba pipeline. Setelah berhasil, ubah menjadi None, restart kernel, lalu Run All untuk seluruh manifest. Mode uji bukan hasil evaluasi penuh.

Buka notebook dari checkout GlucoSight. Model, tabel TKPI, dan raw CGMacros harus tersedia lokal. Tidak ada training, pengunduhan dataset, pengiriman data, atau pembaruan ClickUp dalam notebook ini.

Catatan evaluasi:
- Makro CGMacros dilabeli cgmacros_reported_estimate sesuai data dictionary lokal, bukan otomatis dianggap hasil penimbangan. Jangan kalikan lagi dengan Amount Consumed.
- B0 memakai konstanta repo: 45/18/12/4 g, bukan rata-rata yang dipelajari dari label evaluasi.
- Semua prediksi low_confidence tetap dinilai. Error inference dilaporkan sebagai missing, bukan diubah menjadi nol.
- Confidence adalah softmax mentah; OOD dapat menghasilkan confidence tinggi. Tanpa label kelas yang sepadan, notebook ini tidak menghitung top-1 accuracy atau gap akurasi Indonesia versus CGMacros.
- Hasil berdiri sendiri sebagai evaluasi OOD, terpisah dari tabel utama fusion. Clarke Error Grid berlaku untuk prediksi glukosa; keluaran notebook ini adalah makro dalam gram.
- Notebook sumber disimpan tanpa output. Sebelum commit notebook yang sudah dijalankan, gunakan Clear All Outputs karena tabel/contoh foto dapat memuat data peserta.


## 1. Temukan repo dan siapkan dependency

Jika kernel dibuka di luar checkout, isi REPO_OVERRIDE dengan path repo. Untuk instalasi pertama, ubah INSTALL_DEPENDENCIES menjadi True; instalasi memakai Python kernel aktif. Setelah instalasi, restart kernel bila diminta.


In [ ]:
from pathlib import Path
import sys
import subprocess

REPO_OVERRIDE = None  # contoh Windows: r"D:\Project\GlucoSight"
INSTALL_DEPENDENCIES = False

def find_repo(override):
    if override is not None:
        candidate = Path(override).expanduser().resolve()
        if not (candidate / "cv-baseline-v0.1" / "cv_baseline").is_dir():
            raise FileNotFoundError(f"Bukan checkout GlucoSight: {candidate}")
        return candidate
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "cv-baseline-v0.1" / "cv_baseline").is_dir():
            return candidate
    raise FileNotFoundError("Repo tidak ditemukan. Isi REPO_OVERRIDE di sel ini.")

REPO_ROOT = find_repo(REPO_OVERRIDE)
PACKAGE_ROOT = REPO_ROOT / "cv-baseline-v0.1"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))
if INSTALL_DEPENDENCIES:
    if sys.prefix == sys.base_prefix:
        raise RuntimeError("Pilih kernel GlucoSight CV (.venv) sebelum instalasi; "
                           "Python global dapat dipakai paket lain yang membutuhkan NumPy 2.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-r", str(PACKAGE_ROOT / "requirements-notebook.txt"),
    ])

print("Python:", sys.executable)
print("Repo  :", REPO_ROOT)


In [ ]:
import json
import hashlib
import importlib.metadata
import math
import platform
import time
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from cv_baseline import predict, validate_output
from cv_baseline.classifier import MobileNetV2Classifier
from cv_baseline.predict import set_classifier, get_macro_lookup
from cv_baseline.config import (
    MODEL_PATH, MACRO_TABLE_PATH, MODEL_VERSION, CONFIDENCE_THRESHOLD,
)
from cv_baseline.mean_baseline import predict_population_mean
from cv_baseline.b0_evaluation import (
    MACROS, evaluate_population_mean, write_json, write_jsonl,
)
from cv_baseline.cgmacros_manifest import build_meal_manifest

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_columns", 20)
def check_numpy_torch_bridge(torch_module, numpy_module):
    """Uji kedua arah konversi yang dipakai classifier sebelum memproses foto."""
    try:
        probe = numpy_module.zeros((1,), dtype=numpy_module.float32)
        restored = torch_module.from_numpy(probe).cpu().numpy()
        numpy_module.testing.assert_array_equal(restored, probe)
    except Exception as exc:
        raise RuntimeError(
            f"Konversi NumPy–PyTorch gagal (NumPy {numpy_module.__version__}, "
            f"PyTorch {torch_module.__version__}): {exc}. "
            'Untuk environment baseline PyTorch 2.2.x, jalankan '
            '%pip install "numpy>=1.26.4,<2" pada sel baru, lalu '
            'Restart Kernel dan Run All. Mengulang sel MAE saja tidak cukup.'
        ) from exc

import torch
check_numpy_torch_bridge(torch, np)
print(f"Import dan konversi NumPy–PyTorch berhasil: {np.__version__} / {torch.__version__}")


## 2. Konfigurasi run

CGMACROS_ROOT harus langsung berisi folder CGMacros-001, CGMacros-002, dan seterusnya. None mencoba menemukannya di data/cgmacros/raw. Manifest B0 yang ada selalu diprioritaskan; bila belum ada, parser B0 yang sama dipakai membangunnya di memori.

Mode uji mengambil sampel acak deterministik sebelum melihat prediksi. Jangan mengganti seed/subset setelah melihat angka untuk memilih hasil terbaik. Setelah mengubah konfigurasi, restart kernel dan jalankan semua sel.


In [ ]:
CGMACROS_ROOT = None  # atau Path(r"D:\...\CGMacros") / Path("/path/to/CGMacros")
B0_MANIFEST_PATH = REPO_ROOT / "data/cgmacros/results/b0_population_mean/meal_manifest.jsonl"
B0_METRICS_PATH = PACKAGE_ROOT / "reports/b0_population_mean/metrics.json"

MAX_MEALS = None       # None = evaluasi penuh; bilangan positif = uji pipeline
SEED = 42
DEVICE = "auto"      # "auto", "cpu", atau "cuda"
CPU_THREADS = 4      # membatasi overhead CPU untuk inference satu foto per panggilan
SHOW_EXAMPLES = True  # True menampilkan enam contoh error karbohidrat terbesar

def locate_dataset(explicit_root):
    if explicit_root is not None:
        root = Path(explicit_root).expanduser().resolve()
        if not root.is_dir() or not any(
            p.is_dir() for p in root.glob("CGMacros-[0-9][0-9][0-9]")
        ):
            raise FileNotFoundError(f"Folder peserta CGMacros tidak ditemukan di {root}")
        return root
    raw_root = REPO_ROOT / "data/cgmacros/raw"
    candidates = sorted({
        p.parent.resolve() for p in raw_root.rglob("CGMacros-001") if p.is_dir()
    }) if raw_root.is_dir() else []
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Ditemukan {len(candidates)} kandidat dataset: {candidates}. "
            "Isi CGMACROS_ROOT secara eksplisit."
        )
    return candidates[0]

DATASET_ROOT = locate_dataset(CGMACROS_ROOT)
if MAX_MEALS is not None and (type(MAX_MEALS) is not int or MAX_MEALS < 1):
    raise ValueError("MAX_MEALS harus None atau bilangan bulat positif.")
if type(CPU_THREADS) is not int or CPU_THREADS < 1:
    raise ValueError("CPU_THREADS harus bilangan bulat positif.")
print("Dataset:", DATASET_ROOT)
print("Mode   :", "FULL" if MAX_MEALS is None else f"TRIAL, paling banyak {MAX_MEALS} meal")


## 3. Validasi manifest dan acuan B0

Satu meal hanya memakai before_image_path. meal_id harus unik. Jalur foto relatif terhadap dataset root; file yang hilang dicatat, lalu tetap dicoba agar kegagalannya terlihat dalam audit inference. Bila hash manifest berbeda dari laporan B0 tersimpan, notebook berhenti agar perubahan cohort ditinjau terlebih dahulu.


In [ ]:
manifest_build_summary = None
if B0_MANIFEST_PATH.is_file():
    with B0_MANIFEST_PATH.open(encoding="utf-8-sig") as handle:
        all_records = [json.loads(line) for line in handle if line.strip()]
    manifest_source = "existing_b0_manifest"
else:
    built = build_meal_manifest(DATASET_ROOT)
    all_records = built.records
    manifest_build_summary = built.summary()
    manifest_source = "rebuilt_with_b0_parser"

all_records = sorted(all_records, key=lambda r: (r["participant_id"], r["timestamp"]))
if not all_records:
    raise ValueError("Manifest tidak memiliki meal yang eligible.")
required = {
    "meal_id", "participant_id", "timestamp", "meal_type", "before_image_path",
    "ground_truth_provenance", "source_dataset", "dataset_version", "dataset_license",
    *[f"true_{m}" for m in MACROS],
}
ids = []
image_paths = {}
for record in all_records:
    missing = required - record.keys()
    if missing:
        raise ValueError(f"Field manifest hilang: {missing}")
    ids.append(record["meal_id"])
    if not isinstance(record["meal_id"], str) or not record["meal_id"].strip():
        raise ValueError("meal_id harus string non-kosong.")
    if not record["participant_id"]:
        raise ValueError("participant_id kosong.")
    if record["ground_truth_provenance"] != "cgmacros_reported_estimate":
        raise ValueError("Provenance manifest berbeda; tinjau sebelum mengevaluasi.")
    values = np.asarray([record[f"true_{m}"] for m in MACROS], dtype=float)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError(f"Label makro tidak valid: {record['meal_id']}")
    relative_path = Path(record["before_image_path"])
    if relative_path.is_absolute():
        raise ValueError("before_image_path harus relatif terhadap DATASET_ROOT.")
    resolved = (DATASET_ROOT / relative_path).resolve()
    if not resolved.is_relative_to(DATASET_ROOT):
        raise ValueError("Jalur gambar keluar dari DATASET_ROOT.")
    image_paths[record["meal_id"]] = resolved

if len(set(ids)) != len(ids):
    raise ValueError("meal_id duplikat dalam manifest.")

_, b0_full_metrics = evaluate_population_mean(all_records)
reference_status = "no_saved_b0_metrics"
if B0_METRICS_PATH.is_file():
    saved_b0 = json.loads(B0_METRICS_PATH.read_text(encoding="utf-8-sig"))
    if saved_b0["dataset"]["manifest_sha256"] != b0_full_metrics["dataset"]["manifest_sha256"]:
        raise ValueError(
            "Hash manifest berbeda dari laporan B0. Pastikan manifest dan laporan "
            "berasal dari run yang sama; jangan mengabaikan mismatch."
        )
    for macro in MACROS:
        if not math.isclose(
            saved_b0["pooled"][macro]["mae_g"],
            b0_full_metrics["pooled"][macro]["mae_g"], rel_tol=1e-9, abs_tol=1e-9,
        ):
            raise ValueError(f"Angka B0 berubah pada {macro}; tinjau baseline.")
    reference_status = "manifest_hash_and_b0_mae_match"

manifest_df = pd.DataFrame(all_records)
print(f"Manifest: {len(all_records):,} meal / {manifest_df.participant_id.nunique()} peserta")
print("Foto tidak ditemukan:", sum(not p.is_file() for p in image_paths.values()))
print("Sumber manifest:", manifest_source)
print("Verifikasi B0 :", reference_status)
print("Hash manifest :", b0_full_metrics["dataset"]["manifest_sha256"])
display(pd.DataFrame([
    {"macro": m, "B0_MAE_g_full_manifest": b0_full_metrics["pooled"][m]["mae_g"]}
    for m in MACROS
]).round(3))


## 4. Pilih cohort dan muat model sungguhan

B0 dan classifier akan dinilai pada ID yang sama. Checkpoint, tabel lookup, dan kode inference dicatat dengan SHA-256. Model dibekukan dalam mode eval. Notebook gagal lebih awal jika PyTorch, checkpoint, atau kelas lookup tidak tersedia; tidak ada fallback ke mock.


In [ ]:
is_full_run = MAX_MEALS is None
if is_full_run or MAX_MEALS >= len(all_records):
    selected_records = list(all_records)
else:
    indexes = np.sort(np.random.default_rng(SEED).choice(
        len(all_records), size=MAX_MEALS, replace=False,
    ))
    selected_records = [all_records[int(i)] for i in indexes]

_, b0_selected_metrics = evaluate_population_mean(selected_records)
run_label = "FULL OOD" if is_full_run else "TRIAL ONLY — bukan evaluasi penuh"
print(run_label)
print(f"Terpilih: {len(selected_records)} meal / "
      f"{len({r['participant_id'] for r in selected_records})} peserta")

import torch
if DEVICE not in {"auto", "cpu", "cuda"}:
    raise ValueError("DEVICE harus auto, cpu, atau cuda.")
device = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
if device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA tidak tersedia di kernel ini; pilih cpu atau auto.")
torch.set_num_threads(CPU_THREADS)

for required_file in (MODEL_PATH, MACRO_TABLE_PATH):
    if not required_file.is_file():
        raise FileNotFoundError(f"File lokal wajib belum tersedia: {required_file}")
classifier = MobileNetV2Classifier(device=device).load()
lookup = get_macro_lookup()
missing_classes = set(classifier.class_to_idx) - set(lookup.table)
if missing_classes:
    raise ValueError(f"Kelas tidak memiliki lookup makro: {sorted(missing_classes)}")
set_classifier(classifier)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

code_names = [
    "classifier.py", "predict.py", "preprocessing.py", "config.py",
    "macro_lookup.py", "mean_baseline.py", "cgmacros_manifest.py",
    "b0_evaluation.py", "schema.py",
]
runtime_versions = {}
for package in ("numpy", "pandas", "matplotlib", "torch", "torchvision", "Pillow"):
    runtime_versions[package] = importlib.metadata.version(package)

run_metadata = {
    "evaluation": "out_of_domain",
    "run_label": run_label,
    "is_full_manifest_run": is_full_run,
    "max_meals": MAX_MEALS,
    "selection_seed": SEED,
    "manifest_source": manifest_source,
    "b0_reference_check": reference_status,
    "full_manifest_sha256": b0_full_metrics["dataset"]["manifest_sha256"],
    "selected_manifest_sha256": b0_selected_metrics["dataset"]["manifest_sha256"],
    "checkpoint_sha256": sha256_file(MODEL_PATH),
    "macro_table_sha256": sha256_file(MACRO_TABLE_PATH),
    "inference_code_sha256": {
        name: sha256_file(PACKAGE_ROOT / "cv_baseline" / name) for name in code_names
    },
    "model_version": MODEL_VERSION,
    "confidence_type": "uncalibrated_top1_softmax",
    "low_confidence_threshold": CONFIDENCE_THRESHOLD,
    "prediction_carbs_source": "class_lookup",
    "prediction_fiber_source": "estimated_from_carb_ratio",
    "ground_truth_provenance": "cgmacros_reported_estimate",
    "device": device,
    "cpu_threads": CPU_THREADS,
    "python_version": platform.python_version(),
    "packages": runtime_versions,
    "full_manifest_meals": len(all_records),
    "selected_meals": len(selected_records),
    "selected_participants": len({r["participant_id"] for r in selected_records}),
}
print("Model:", MODEL_VERSION, "| device:", device)
print("Kelas:", sorted(classifier.class_to_idx))
print("Checkpoint SHA-256:", run_metadata["checkpoint_sha256"])


## 5. Inference dan penyimpanan hasil mentah

Setiap run membuat direktori baru di data/cgmacros/results/indonesian_classifier_ood (diabaikan Git). Prediksi ditulis satu per satu; bila kernel terputus, file parsial tetap tersedia tetapi tidak dianggap hasil lengkap. Jalankan ulang sel ini untuk run baru. Angka makro referensi tidak dikirim ke fungsi predict.


In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = REPO_ROOT / "data/cgmacros/results/indonesian_classifier_ood" / (
    ("full_" if is_full_run else "trial_") + stamp
)
RUN_DIR.mkdir(parents=True, exist_ok=False)
REPORT_DIR = (
    REPO_ROOT / "cv/reports/indonesian_classifier_ood" / stamp
    if is_full_run else RUN_DIR / "report"
)
REPORT_DIR.mkdir(parents=True, exist_ok=False)
write_jsonl(RUN_DIR / "selected_manifest.jsonl", selected_records)
write_json(RUN_DIR / "manifest_build_summary.json", manifest_build_summary)

run_metadata["started_at_utc"] = datetime.now(timezone.utc).isoformat()
run_metadata["inference_complete"] = False
write_json(RUN_DIR / "run_metadata.json", run_metadata)

prediction_rows = []
started = time.perf_counter()
with (RUN_DIR / "predictions.jsonl").open("w", encoding="utf-8") as handle:
    for index, record in enumerate(selected_records, start=1):
        prediction = predict({
            "sample_id": record["meal_id"],
            "meal_image_path": str(image_paths[record["meal_id"]]),
        })
        if prediction.get("feature_status") == "mock":
            raise RuntimeError("Output mock terdeteksi; evaluasi dibatalkan.")
        if prediction.get("sample_id") != record["meal_id"]:
            raise RuntimeError("ID output berbeda dari ID input.")
        non_finite = [
            field for field in (*MACROS, "confidence", "calories_kcal")
            if isinstance(prediction.get(field), (int, float))
            and not math.isfinite(prediction[field])
        ]
        if non_finite:
            prediction = dict(prediction)
            prediction["feature_status"] = "error"
            prediction["error_reason"] = "non_finite_output:" + ",".join(non_finite)
            for field in non_finite:
                prediction[field] = None
        prediction_rows.append(prediction)
        handle.write(json.dumps(prediction, ensure_ascii=False, allow_nan=False) + "\n")
        handle.flush()
        if index == 1 or index % 100 == 0 or index == len(selected_records):
            elapsed = time.perf_counter() - started
            print(f"{index}/{len(selected_records)} | {elapsed:.1f} s | "
                  f"{prediction.get('feature_status')}")

run_metadata["inference_seconds"] = time.perf_counter() - started
run_metadata["inference_complete"] = len(prediction_rows) == len(selected_records)
run_metadata["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
write_json(RUN_DIR / "run_metadata.json", run_metadata)
print("Status:", dict(Counter(r.get("feature_status", "missing") for r in prediction_rows)))
print("Output:", RUN_DIR)


## 6. Join berdasarkan ID, audit error, dan hitung metrik berpasangan

MAE dihitung per makro pada semua prediksi yang valid, termasuk confidence rendah. Baris error, schema invalid, atau nilai non-finite tidak dinilai; cakupan dan alasannya dilaporkan. B0 dihitung ulang pada subset valid yang persis sama. Median/IQR dihitung dari MAE per peserta, sehingga setiap peserta mempunyai bobot yang sama pada ringkasan tersebut.


In [ ]:
def prepare_scored_records(records, predictions):
    """Join one-to-one; jangan menganggap validasi schema == inference berhasil."""
    expected_ids = [r["meal_id"] for r in records]
    actual_ids = [r.get("sample_id") for r in predictions]
    if (len(expected_ids) != len(set(expected_ids))
            or len(actual_ids) != len(set(actual_ids))
            or set(expected_ids) != set(actual_ids)):
        raise ValueError("ID manifest/prediksi duplikat, hilang, atau tidak cocok.")
    if any(r.get("feature_status") == "mock" for r in predictions):
        raise ValueError("Prediksi mock tidak boleh dinilai.")

    joined_rows = []
    by_id = {r["sample_id"]: r for r in predictions}
    for record in records:
        prediction = by_id[record["meal_id"]]
        reasons = list(validate_output(prediction))
        if prediction.get("feature_status") not in {"ok", "low_confidence"}:
            reasons.append(prediction.get("error_reason") or "inference_not_valid")
        for field in (*MACROS, "confidence"):
            value = prediction.get(field)
            try:
                finite = value is not None and math.isfinite(float(value))
            except (TypeError, ValueError):
                finite = False
            if not finite:
                reasons.append(f"non_finite_or_missing:{field}")
        joined_rows.append({
            "meal_id": record["meal_id"],
            "participant_id": record["participant_id"],
            "meal_type": record["meal_type"],
            "before_image_path": record["before_image_path"],
            **{f"true_{m}": record[f"true_{m}"] for m in MACROS},
            **{f"pred_{m}": prediction.get(m) for m in MACROS},
            "confidence": prediction.get("confidence"),
            "food_top1": prediction.get("food_top1"),
            "feature_status": prediction.get("feature_status"),
            "scored": not reasons,
            "exclusion_reason": "; ".join(dict.fromkeys(reasons)),
        })
    joined = pd.DataFrame(joined_rows)
    scored = joined.loc[joined["scored"]].copy()
    audit = joined.loc[~joined["scored"]].copy()
    for field in [*[f"pred_{m}" for m in MACROS], "confidence"]:
        scored[field] = pd.to_numeric(scored[field], errors="raise")
    return scored, audit, joined

def score_macros(scored):
    if scored.empty:
        raise ValueError("Tidak ada inference valid; periksa audit_errors.csv.")
    fixed_b0 = predict_population_mean(str(scored.iloc[0]["meal_id"]))
    paired = scored.copy()
    pooled_rows, participant_rows = [], []
    for macro in MACROS:
        paired[f"b0_{macro}"] = float(fixed_b0[macro])
        paired[f"cv_ae_{macro}"] = np.abs(
            paired[f"pred_{macro}"].to_numpy(dtype=float)
            - paired[f"true_{macro}"].to_numpy(dtype=float)
        )
        paired[f"b0_ae_{macro}"] = np.abs(
            paired[f"b0_{macro}"].to_numpy(dtype=float)
            - paired[f"true_{macro}"].to_numpy(dtype=float)
        )
        cv_errors = paired[f"cv_ae_{macro}"]
        b0_errors = paired[f"b0_ae_{macro}"]
        pooled_rows.append({
            "macro": macro, "n_meals": len(paired),
            "cv_mae_g": float(cv_errors.mean()), "b0_mae_g": float(b0_errors.mean()),
            "delta_mae_g": float(cv_errors.mean() - b0_errors.mean()),
            "cv_rmse_g": float(np.sqrt(np.mean(cv_errors**2))),
            "b0_rmse_g": float(np.sqrt(np.mean(b0_errors**2))),
        })
        for participant_id, group in paired.groupby("participant_id"):
            cv_mae = float(group[f"cv_ae_{macro}"].mean())
            b0_mae = float(group[f"b0_ae_{macro}"].mean())
            participant_rows.append({
                "participant_id": participant_id, "macro": macro, "n_meals": len(group),
                "cv_mae_g": cv_mae, "b0_mae_g": b0_mae, "delta_mae_g": cv_mae - b0_mae,
            })
    per_participant = pd.DataFrame(participant_rows)
    summary_rows = []
    for macro in MACROS:
        group = per_participant.loc[per_participant["macro"] == macro]
        row = {"macro": macro, "n_participants": len(group)}
        for method in ("cv", "b0", "delta"):
            values = group[f"{method}_mae_g"]
            row[f"{method}_median_mae_g"] = float(values.median())
            row[f"{method}_q1_mae_g"] = float(values.quantile(0.25))
            row[f"{method}_q3_mae_g"] = float(values.quantile(0.75))
        summary_rows.append(row)
    return paired, pd.DataFrame(pooled_rows), per_participant, pd.DataFrame(summary_rows)


In [ ]:
scored, audit, joined = prepare_scored_records(selected_records, prediction_rows)
joined.to_csv(RUN_DIR / "joined_predictions.csv", index=False)
audit.to_csv(RUN_DIR / "audit_errors.csv", index=False)
coverage = {
    "full_manifest_meals": len(all_records),
    "selected_meals": len(selected_records),
    "scored_meals": len(scored),
    "failed_meals": len(audit),
    "scored_fraction": len(scored) / len(selected_records),
    "selected_participants": len({r["participant_id"] for r in selected_records}),
    "scored_participants": int(scored["participant_id"].nunique()),
    "prediction_status_counts": dict(Counter(r["feature_status"] for r in prediction_rows)),
    "failure_reason_counts": dict(Counter(audit["exclusion_reason"])),
}
write_json(RUN_DIR / "coverage.json", coverage)
display(pd.DataFrame([coverage]).drop(columns=["prediction_status_counts", "failure_reason_counts"]))
if not audit.empty:
    print("Ada kegagalan. B0 di bawah hanya memakai meal yang berhasil dinilai.")
    display(audit[["meal_id", "feature_status", "exclusion_reason"]].head(10))

if scored.empty:
    reasons = Counter(
        p.get("error_reason") or f"schema/status invalid: {p.get('feature_status')}"
        for p in prediction_rows
    )
    detail = "; ".join(f"{count} meal: {reason}" for reason, count in reasons.most_common(3))
    raise ValueError(
        f"Semua {len(prediction_rows)} inference gagal. {detail}. "
        f"Audit: {RUN_DIR / 'audit_errors.csv'}. Perbaiki penyebab inference, "
        "lalu Restart Kernel dan Run All; jangan mengganti prediksi gagal dengan nol."
    )

paired, comparison, per_participant, participant_summary = score_macros(scored)

# Cross-check dengan evaluator B0 repo, memakai ID valid yang sama.
valid_ids = set(scored["meal_id"])
_, paired_b0_reference = evaluate_population_mean([
    r for r in selected_records if r["meal_id"] in valid_ids
])
for row in comparison.to_dict("records"):
    assert math.isclose(
        row["b0_mae_g"], paired_b0_reference["pooled"][row["macro"]]["mae_g"],
        rel_tol=1e-12, abs_tol=1e-12,
    )

display(Markdown(f"**{run_label}** — delta MAE negatif berarti classifier lebih baik dari B0."))
display(comparison.round(3))
display(Markdown("**Median dan IQR MAE antar peserta** (unit gram)"))
display(participant_summary.round(3))
print("Tidak ada prediksi low_confidence yang dibuang berdasarkan threshold.")


## 7. Confidence dan sebaran kelas

Histogram berikut mendeskripsikan keyakinan model pada OOD, bukan akurasi atau kalibrasi. Threshold hanya dipakai untuk ringkasan, tidak untuk memilih meal yang dinilai. Confidence dibulatkan empat desimal oleh predict.py; hitungan status low_confidence diambil langsung dari output model untuk menghindari perbedaan pembulatan di batas threshold.


In [ ]:
confidence = paired["confidence"].astype(float)
hist_counts, hist_edges = np.histogram(confidence, bins=np.linspace(0, 1, 11))
confidence_bins = pd.DataFrame({
    "lower_bound": hist_edges[:-1], "upper_bound": hist_edges[1:],
    "n_meals": hist_counts, "fraction": hist_counts / len(paired),
})
confidence_summary = {
    "n_meals": len(paired),
    "mean": float(confidence.mean()), "median": float(confidence.median()),
    "q1": float(confidence.quantile(0.25)), "q3": float(confidence.quantile(0.75)),
    "min": float(confidence.min()), "max": float(confidence.max()),
    "fraction_status_low_confidence": float((paired.feature_status == "low_confidence").mean()),
    "fraction_reported_confidence_ge_0_9": float((confidence >= 0.9).mean()),
    "threshold": CONFIDENCE_THRESHOLD,
    "type": "uncalibrated_top1_softmax_rounded_4dp",
}
class_distribution = (
    paired["food_top1"].value_counts().rename_axis("predicted_class")
    .reset_index(name="n_meals")
)
display(pd.DataFrame([confidence_summary]))
display(confidence_bins.round(3))
display(class_distribution)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(confidence, bins=hist_edges, color="#167d8d", edgecolor="white")
axes[0].axvline(CONFIDENCE_THRESHOLD, color="#b44a3c", linestyle="--",
                label=f"Threshold {CONFIDENCE_THRESHOLD:.2f}")
axes[0].set(xlim=(0, 1), xlabel="Top-1 softmax (belum dikalibrasi)",
            ylabel="Jumlah meal", title="Confidence pada foto OOD")
axes[0].legend()
x = np.arange(len(comparison))
axes[1].bar(x - 0.19, comparison["b0_mae_g"], width=0.38, label="B0", color="#9ca6b5")
axes[1].bar(x + 0.19, comparison["cv_mae_g"], width=0.38, label="Classifier", color="#167d8d")
axes[1].set_xticks(x, comparison["macro"])
axes[1].set(ylabel="MAE (gram)", title="Meal yang sama; MAE lebih rendah lebih baik")
axes[1].legend()
fig.suptitle(f"{run_label} | n={len(paired)} meal, {coverage['scored_participants']} peserta")
fig.tight_layout()
fig.savefig(REPORT_DIR / "ood_summary.png", dpi=160, bbox_inches="tight")
plt.show()


## 8. Inspeksi contoh, opsional

Aktifkan SHOW_EXAMPLES pada konfigurasi untuk melihat enam error karbohidrat terbesar. Pemilihan ini hanya untuk diagnosis; semua meal valid tetap masuk metrik. Foto hanya ditampilkan lokal dan tidak disalin ke laporan.


In [ ]:
if SHOW_EXAMPLES:
    from PIL import Image
    examples = paired.nlargest(min(6, len(paired)), "cv_ae_carbs_g")
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    for ax in axes.flat:
        ax.axis("off")
    for ax, row in zip(axes.flat, examples.to_dict("records")):
        with Image.open(image_paths[row["meal_id"]]) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(
            f"{row['food_top1']} | confidence={row['confidence']:.2f}\n"
            f"Carbs pred={row['pred_carbs_g']:.1f} g / label={row['true_carbs_g']:.1f} g\n"
            f"Absolute error={row['cv_ae_carbs_g']:.1f} g",
            fontsize=9,
        )
    fig.suptitle("Contoh error terbesar — diagnosis, tanpa mengubah cohort")
    fig.tight_layout()
    plt.show()
else:
    print("Contoh foto dinonaktifkan. Ubah SHOW_EXAMPLES=True untuk melihatnya.")


## 9. Ekspor hasil, laporan singkat, dan draft model card

Mode FULL menulis ringkasan agregat ke cv/reports/indonesian_classifier_ood/<run-id>. Mode TRIAL menyimpan laporan di direktori hasil lokal dan memberi label percobaan. Detail per meal/peserta selalu disimpan di data/ yang diabaikan Git. MODEL_CARD_OOD_snippet.md adalah draft untuk ditinjau dan disalin ke model card setelah full run; notebook tidak mengubah MODEL_CARD.md secara otomatis.

Perbandingan ini mengukur performa pipeline pada cohort OOD. Delta MAE terhadap B0 bukan estimasi kausal penurunan performa akibat domain shift; variasi porsi, label, dan lookup juga berkontribusi.


In [ ]:
def markdown_table(frame, columns, decimals=3):
    """Tabel tanpa dependency tabulate; kolom hasil bersifat agregat."""
    def fmt(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.{decimals}f}"
        return str(value).replace("|", "/").replace("\n", " ")
    lines = ["| " + " | ".join(columns) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    for row in frame[columns].itertuples(index=False, name=None):
        lines.append("| " + " | ".join(fmt(value) for value in row) + " |")
    return "\n".join(lines)

# Konsistensi artifact: jangan ekspor jika variabel konfigurasi/model berubah setelah inference.
if (run_metadata["selected_meals"] != len(selected_records)
        or run_metadata["is_full_manifest_run"] != is_full_run
        or run_metadata["max_meals"] != MAX_MEALS
        or run_metadata["selection_seed"] != SEED
        or sha256_file(MODEL_PATH) != run_metadata["checkpoint_sha256"]
        or sha256_file(MACRO_TABLE_PATH) != run_metadata["macro_table_sha256"]):
    raise RuntimeError("Konfigurasi/model berubah. Restart kernel dan jalankan semua sel.")
if any(sha256_file(PACKAGE_ROOT / "cv_baseline" / name) != digest
       for name, digest in run_metadata["inference_code_sha256"].items()):
    raise RuntimeError("Kode inference berubah di tengah run; ulangi dari kernel baru.")

paired.to_csv(RUN_DIR / "scored_predictions.csv", index=False)
per_participant.to_csv(RUN_DIR / "per_participant_metrics.csv", index=False)
comparison.to_csv(REPORT_DIR / "macro_comparison.csv", index=False)
participant_summary.to_csv(REPORT_DIR / "participant_summary.csv", index=False)
confidence_bins.to_csv(REPORT_DIR / "confidence_distribution.csv", index=False)
class_distribution.to_csv(REPORT_DIR / "predicted_class_distribution.csv", index=False)

metrics = {
    "metadata": run_metadata,
    "coverage": {k: v for k, v in coverage.items() if k != "failure_reason_counts"},
    "b0_fixed_prediction": {m: predict_population_mean("report")[m] for m in MACROS},
    "comparison": comparison.to_dict("records"),
    "participant_summary": participant_summary.to_dict("records"),
    "confidence": confidence_summary,
    "confidence_histogram": confidence_bins.to_dict("records"),
    "predicted_classes": class_distribution.to_dict("records"),
}
# Ketat terhadap NaN/Infinity; gagal daripada menyimpan metrik non-standar.
(REPORT_DIR / "metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8",
)

wins = comparison.loc[comparison.delta_mae_g < 0, "macro"].tolist()
win_text = ", ".join(wins) if wins else "tidak ada makro"
interpretation = (
    f"Pada {len(paired)} meal dari {coverage['scored_participants']} peserta "
    f"({coverage['scored_fraction']:.1%} dari meal terpilih berhasil dinilai), "
    f"MAE classifier lebih rendah dari B0 pada {win_text}. "
    f"Median confidence adalah {confidence_summary['median']:.3f}, tetapi softmax ini "
    "belum dikalibrasi dan confidence tinggi tidak membuktikan prediksi OOD benar. "
    "Angka ini menilai classifier dengan lookup/porsi tetap terhadap reported estimates "
    "CGMacros, termasuk serat heuristik, dan dipisahkan dari evaluasi utama fusion."
)
comparison_md = markdown_table(comparison, [
    "macro", "n_meals", "cv_mae_g", "b0_mae_g", "delta_mae_g", "cv_rmse_g", "b0_rmse_g",
])
participant_display = participant_summary[["macro", "n_participants"]].copy()
for method in ("cv", "b0", "delta"):
    participant_display[f"{method}_median_[q1,q3]_g"] = [
        f"{r[f'{method}_median_mae_g']:.3f} "
        f"[{r[f'{method}_q1_mae_g']:.3f}, {r[f'{method}_q3_mae_g']:.3f}]"
        for r in participant_summary.to_dict("records")
    ]

mode_note = (
    "Seluruh manifest dicoba. Periksa cakupan inference dan error sebelum memakai hasil."
    if is_full_run else
    "**TRIAL ONLY: subset untuk mencoba pipeline, bukan hasil akhir task Sprint 2.**"
)
report = f"""# Indonesian classifier on CGMacros — out-of-domain

{mode_note}

- Run: {stamp}; model: {MODEL_VERSION}; device: {device}
- Manifest penuh: {len(all_records)} meal; terpilih: {len(selected_records)}
- Dinilai: {len(paired)} meal / {coverage['scored_participants']} peserta
- Gagal dinilai: {len(audit)}; coverage: {coverage['scored_fraction']:.1%}
- Ground-truth provenance: cgmacros_reported_estimate
- Prediction provenance: class_lookup; fiber: estimated_from_carb_ratio
- B0: carbs/protein/fat/fiber = 45/18/12/4 g, tetap
- Hash manifest penuh: {run_metadata['full_manifest_sha256']}
- Hash manifest terpilih: {run_metadata['selected_manifest_sha256']}
- Hash checkpoint: {run_metadata['checkpoint_sha256']}
- Hash tabel makro: {run_metadata['macro_table_sha256']}

## Per-macro error

B0 dan classifier dinilai pada meal valid yang persis sama. Semua low_confidence tetap
masuk. Delta = MAE classifier dikurangi MAE B0; negatif berarti classifier lebih baik.

{comparison_md}

## MAE per peserta: median [Q1, Q3]

{markdown_table(participant_display, list(participant_display.columns))}

## Confidence

- Jenis: softmax top-1 mentah, dibulatkan 4 desimal oleh predict.py
- Mean: {confidence_summary['mean']:.3f}; median: {confidence_summary['median']:.3f}
- Q1–Q3: {confidence_summary['q1']:.3f}–{confidence_summary['q3']:.3f}
- Status low_confidence (threshold {CONFIDENCE_THRESHOLD}): {confidence_summary['fraction_status_low_confidence']:.1%}
- Reported confidence >= 0.9: {confidence_summary['fraction_reported_confidence_ge_0_9']:.1%}

![Distribusi confidence dan MAE OOD](ood_summary.png)

{markdown_table(confidence_bins, list(confidence_bins.columns))}

## Audit

Jumlah status prediksi: {json.dumps(coverage['prediction_status_counts'], ensure_ascii=False)}.
Detail kegagalan disimpan lokal di audit_errors.csv dalam direktori run di data/.
Tidak ada zero-imputation atau filter berbasis confidence.
Bila coverage < 100%, tabel MAE hanya berlaku untuk subset yang berhasil; ini bukan hasil
classifier pada semua meal eligible. Error preprocessing/inference perlu ditinjau.

## Interpretasi

{interpretation}

## Batas interpretasi

Model selalu memilih satu dari 10 kelas Indonesia. Notebook tidak memiliki anotasi kelas
CGMacros yang sepadan, sehingga tidak melaporkan accuracy atau mengurangkan accuracy
Indonesia dari MAE OOD. Selisih terhadap B0 juga bukan efek kausal domain shift.
Label CGMacros merujuk consumed meal; Amount Consumed tidak dikalikan ulang.
Porsi diasumsikan tetap per kelas dan serat adalah heuristik.
Tidak ada tuning pada cohort ini. Hasil ini tidak masuk tabel utama fusion.
"""
(REPORT_DIR / "report.md").write_text(report, encoding="utf-8")
snippet = f"""## Out-of-domain: CGMacros ({stamp})

{mode_note}

{interpretation}

{comparison_md}

Ground truth: cgmacros_reported_estimate. Prediction: class_lookup; serat heuristik.
Laporan lengkap: cv/reports/indonesian_classifier_ood/{stamp}/report.md
""" if is_full_run else (
    "TRIAL ONLY — jangan salin angka ini sebagai hasil penuh ke MODEL_CARD.md.\n\n"
    + interpretation + "\n\n" + comparison_md + "\n"
)
(REPORT_DIR / "MODEL_CARD_OOD_snippet.md").write_text(snippet, encoding="utf-8")
display(Markdown(report))
print("Prediksi dan audit lokal:", RUN_DIR)
print("Laporan dan metrik      :", REPORT_DIR)
print("Draft model card        :", REPORT_DIR / "MODEL_CARD_OOD_snippet.md")


## Setelah run

1. Jika mode TRIAL berhasil, ubah MAX_MEALS menjadi None, restart kernel, lalu Run All.
2. Tinjau coverage, audit error, empat MAE, histogram confidence, dan ringkasan per peserta.
3. Pertahankan angka buruk sebagai temuan; jangan mengubah cohort/threshold/lookup untuk menyembunyikannya.
4. Tinjau report.md dan draft model card hasil FULL. Salin bagian yang sudah ditinjau ke MODEL_CARD.md.
5. Sebelum commit notebook, bersihkan semua output. Raw foto, manifest, dan prediksi per peserta tetap di data/.

Troubleshooting:
- Numpy is not available dengan PyTorch 2.2.x / NumPy 2.x: jalankan `%pip install "numpy>=1.26.4,<2"` pada kernel ini, lalu **Restart Kernel** dan **Run All**. Sel MAE yang kosong hanya gejala lanjut dari kegagalan inference.
- ModuleNotFoundError: aktifkan INSTALL_DEPENDENCIES, jalankan sel instalasi, lalu restart kernel.
- Checkpoint/TKPI tidak ditemukan: salin aset lokal ke models/mobilenetv2_food10.pt dan data/tkpi_macro_lookup.csv dalam paket CV; tidak ada fallback mock.
- Dataset ambigu: isi CGMACROS_ROOT dengan folder yang langsung berisi peserta.
- Hash B0 berbeda: cocokkan meal_manifest.jsonl dengan metrics.json dari run B0 yang sama.
- Seluruh inference error: baca audit_errors.csv, periksa file foto dan environment; metrik tidak akan dipaksakan.
- Run lambat: coba DEVICE="auto" untuk memakai CUDA bila tersedia; default CPU tetap didukung.
- Jangan mengubah file model/tabel/kode di tengah run. Restart kernel sebelum run baru.
